# Ejercicio Módulo 5 - Dataset Swiss
**Inteligencia Artificial - CEIA - FIUBA**

**Luis Jose Paredes Ramirez**

Para aprender sobre regresión, vamos a utilizar un dataset clásico llamado Swiss, que proviene originalmente del lenguaje R. Este dataset contiene datos socioeconómicos de 47 provincias suizas a fines del siglo XIX. Cada fila representa una provincia, y las variables reflejan características demográficas y sociales relevantes para ese contexto histórico.

## Variables

- `Location`: Provincia donde se midieron los datos.
- `Fertility`: Tasa de fertilidad (número promedio de hijos por mujer)
- `Agriculture`: Porcentaje de hombres ocupados en agricultura
- `Examination`: Porcentaje de hombres que completaron exámenes de educación superior
- `Education`: Nivel promedio de educación (escala arbitraria)
- `Catholic`: Porcentaje de población católica
- `Infant.Mortality`: Tasa de mortalidad infantil (por cada 1000 nacidos vivos)

## Que queremos predecir?

Vamos a utilizar este dataset para predecir la tasa de fertilidad en cada provincia mediante diferentes métodos de regresión.

--- 

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y testeo utilizando Scikit-Learn...

In [1]:
import pandas as pd

df = pd.read_csv("swiss.csv")

df.head()

,Location,Fertility,Agriculture,Examination,Education,Catholic,Infant.Mortality
0,Courtelary,80.2,17.0,15,12,9.96,22.2
1,Delemont,83.1,45.1,6,9,84.84,22.2
2,Franches-Mnt,92.5,39.7,5,5,93.40,20.2
3,Moutier,85.8,36.5,12,7,33.77,20.3
4,Neuveville,76.9,43.5,17,15,5.16,20.6


In [2]:
print(f"Tenemos {df.shape[0]} observaciones")

Tenemos 47 observaciones


Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos (quitamos `Location` ya que no es un atributo numérico relevante para la regresión)

In [3]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

Dado que tenemos pocas observaciones, vamos a separar el dataset en un 50% para entrenamiento y 50% para testeo:

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

## Regresión lineal múltiple

Arranquemos la primera parte del ejercicio. Para eso, vamos a entrenar un modelo de regresión lineal múltiple usando todos los atributos. Para ello debes:

1. Escalar los atributos usando `StandardScaler`
2. Entrenar el modelo usando el dataset de entrenamiento.
3. Obtener las predicciones sobre el dataset de testeo.
4. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pipeline_lr = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

pipeline_lr.fit(X_train, y_train)

y_pred_lr = pipeline_lr.predict(X_test)

mae_lr: float = mean_absolute_error(y_test, y_pred_lr)
mse_lr: float = mean_squared_error(y_test, y_pred_lr)
r2_lr: float = r2_score(y_test, y_pred_lr)

print("Metricas del modelo Regresión Lineal Múltiple")
print("-------------------------")
print(f"MAE: {mae_lr:.4f}")
print(f"MSE: {mse_lr:.4f}")
print(f"R²:  {r2_lr:.4f}")

Metricas del modelo Regresión Lineal Múltiple
-------------------------
MAE: 6.0937
MSE: 64.4572
R²:  0.5744


## Modelo con regularización

Para mejorar nuestro modelo, vamos a explorar técnicas de regresión lineal con regularización, que nos permiten controlar el sobreajuste y seleccionar variables relevantes automáticamente.

Existen dos variantes muy populares:

- Una penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- La otra penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Ambas ayudan a mejorar la generalización, pero una de ellas además puede eliminar variables (coeficientes exactamente cero), lo que ayuda a identificar qué atributos son realmente importantes.

Tu tarea:

1. Elegí correctamente cuál de los dos métodos de regularización usar para este problema. 
    - Pista: Queremos que el modelo sea capaz de hacer una selección automática de variables, dejando fuera aquellas que no aportan.
2. Implementá un pipeline que incluya escalado y el modelo elegido.
3. Buscá automáticamente el mejor valor del hiperparámetro de regularización (alpha) usando validación cruzada usando 3-folds.
4. Entrená el modelo con los datos de entrenamiento y obtené las predicciones para el set de testeo.
5. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.
6. Imprimí los coeficientes resultantes e identificá qué variables fueron eliminadas (coeficiente = 0).

In [6]:
import numpy as np

from sklearn.linear_model import LassoCV, RidgeCV

Paremos un momento para entender qué hacen LassoCV y RidgeCV antes de continuar con la resolución:

> Tanto `LassoCV` como `RidgeCV` son implementaciones de regresión lineal con regularización que incluyen la búsqueda automática del mejor hiperparámetro alpha mediante validación cruzada.
>
> Ambos métodos prueban distintos valores de alpha y eligen el que minimiza el error del modelo, facilitando el proceso de ajuste sin necesidad de una búsqueda manual.
>
> Internamente, utilizan la métrica del error cuadrático medio (MSE) para evaluar el rendimiento del modelo en cada fold de la validación cruzada.
>
> Por ejemplo, si llamás a RidgeCV(alphas=alphas, cv=5), se hará una validación cruzada de 5 folds utilizando los valores de alpha que vos le pases, y se seleccionará el que obtenga el menor MSE promedio.
> 
> Una vez elegido el mejor alpha, el modelo final se entrena con todos los datos de entrenamiento usando ese valor.

¡Listo! Con todo lo que vimos hasta ahora, ya estás en condiciones de resolver esta parte y completar los 6 puntos propuestos

In [7]:
alphas = np.logspace(-4, 1, 500)

# Tal como vimos en la clase teorica, para Regresión de Lasso (L1), cuando 𝛼 crece 
# hace que algunos coeficientes se conviertan exactamente en cero.
# Por lo tanto, Lasso realiza una selección automática de atributos.
lasso_cv = LassoCV(alphas=alphas, cv=3, random_state=42)

pipeline_lasso = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('regressor', lasso_cv)
])

pipeline_lasso.fit(X_train, y_train)

y_pred_lasso = pipeline_lasso.predict(X_test)

mae_lasso: float = mean_absolute_error(y_test, y_pred_lasso)
mse_lasso: float = mean_squared_error(y_test, y_pred_lasso)
r2_lasso: float = r2_score(y_test, y_pred_lasso)

print("Regresión Lasso (L1)")
print("--------------------")
print(f"Mejor alpha encontrado: {pipeline_lasso.named_steps['regressor'].alpha_:.6f}")
print(f"MAE: {mae_lasso:.4f}")
print(f"MSE: {mse_lasso:.4f}")
print(f"R²:  {r2_lasso:.4f}")

print("\nCoeficientes del modelo Lasso:")
feature_names = X_train.columns
coeficientes = pipeline_lasso.named_steps['regressor'].coef_

for feat, coef in zip(feature_names, coeficientes):
    feature_eliminated_str = " <- FEATURE ELIMINADA (coef = 0)" if coef == 0.0 else ""
    print(f"  {feat}: {coef:.4f}{feature_eliminated_str}")

Regresión Lasso (L1)
--------------------
Mejor alpha encontrado: 2.181109
MAE: 5.9954
MSE: 64.2943
R²:  0.5755

Coeficientes del modelo Lasso:
  Agriculture: -0.0000 <- FEATURE ELIMINADA (coef = 0)
  Examination: -1.7624
  Education: -2.4049
  Catholic: 1.7196
  Infant.Mortality: 3.2004


## Comparación de modelos y conclusiones

Completá la siguiente tabla con las métricas obtenidas para cada uno de los modelos que entrenaste:


| Modelo             | MAE    | MSE     | R²     |
| ------------------ | ------ | ------- | ------ |
| Regresión Lineal   | 6.0937 | 64.4572 | 0.5744 |
| Lasso (L1)         | 5.9954 | 64.2943 | 0.5755 |

### Justificación

**¿Cuál de los modelos te parece que tuvo un mejor desempeño general?**

El modelo **Lasso** tuvo un desempeño levemente superior en las tres métricas: menor MAE (5.99 vs 6.09), menor MSE (64.29 vs 64.46) y mayor R² (0.5755 vs 0.5744).

Si bien las diferencias son pequeñas y el tamaño reducido del dataset (47 observaciones y aparte hicimos split 50/50), Lasso tiene la ventaja adicional de haber eliminado automáticamente la variable `Agriculture` (coeficiente = 0), produciendo un modelo más simple e interpretable sin sacrificar rendimiento predictivo. 

Esto lo convierte en la opción preferible: igual o mejor precisión con menor complejidad.

In [8]:
comparison = {
    "Linear Regression": {
        "MAE": mae_lr,
        "MSE": mse_lr,
        "R2": r2_lr
    },
    "Lasso": {
        "MAE": mae_lasso,
        "MSE": mse_lasso,
        "R2": r2_lasso
    }
}

pd.DataFrame.from_dict(comparison).T

,MAE,MSE,R2
Linear Regression,6.093707,64.457194,0.574446
Lasso,5.995405,64.294318,0.575522
